# Large Strict No-MCI Direct-Flow Rich Analysis

This notebook analyzes the current frozen no-skip SIREN direct-flow experiment. It does not compare with BrainODE. Numeric volume/trend analysis uses 50 balanced subjects per split. Detailed 3D shape pages are generated only for representative examples.

In [ ]:
from pathlib import Path
import sys
from importlib import reload

import pandas as pd
from IPython.display import Markdown, display

def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / '.git').exists() or (candidate / 'deep_sdf').exists():
            return candidate
    raise RuntimeError(f'Could not find repo root from {start}')

REPO = find_repo_root(Path.cwd())
EXPERIMENT = (REPO / 'examples' / 'ADNI_1_L_No_MCI_large_strict_left' / 'task3_longitudinal_prediction' / 'siren_no_skip_flow_real_sdf_observed_virtual_cocycle').resolve()
CHECKPOINT = 'best'
DEVICE = 'auto'

SELECTED_SUBJECTS_PER_SPLIT = 50
MIN_SCANS_PER_SUBJECT = 3

BATCH_MESH_RESOLUTION = 80
CASE_MESH_RESOLUTION = 96
COUNTERFACTUAL_MESH_RESOLUTION = 80
MESH_MAX_BATCH = 2 ** 18

OOD_AGES = [92.0, 95.0, 100.0, 105.0]
COMPOSED_STEP_YEARS = 0.5
COUNTERFACTUAL_HORIZON_YEARS = 10.0
COUNTERFACTUAL_STEP_YEARS = 2.0
CHANGE_SAMPLE_COUNT = 3000

RUN_BATCH_VOLUME_DECODE = True
RUN_REPRESENTATIVE_SHAPES = True
RUN_MCI_REFERENCE = True
CLEAR_PREVIOUS_NOTEBOOK_HTML = False
INLINE_FIGURE_LIMIT = 8

for extra in [
    REPO,
    REPO / 'examples' / 'ADNI_1_L_No_MCI',
    REPO / 'examples' / 'ADNI_1_L_With_MCI',
    EXPERIMENT / 'scripts',
]:
    if str(extra) not in sys.path:
        sys.path.insert(0, str(extra))

display(Markdown(f'Experiment: `{EXPERIMENT}`'))


In [ ]:
import direct_flow_rich_notebook_helpers as direct_helpers
import rich_visualization_helpers as rich
import adni_original_speed_helpers as mci_helpers

direct_helpers = reload(direct_helpers)
rich = reload(rich)
mci_helpers = reload(mci_helpers)

bundle = direct_helpers.load_bundle(EXPERIMENT, checkpoint=CHECKPOINT, device=DEVICE)
if CLEAR_PREVIOUS_NOTEBOOK_HTML:
    removed_html = direct_helpers.clear_generated_html(bundle)
else:
    removed_html = []

display(Markdown(f'Checkpoint: `{bundle.checkpoint_name}` epoch `{bundle.checkpoint["epoch"]}`'))
display(Markdown(f'Decoder epoch: `{bundle.decoder_epoch}`'))
display(Markdown(f'Device: `{bundle.device}`'))
display(Markdown(f'HTML output: `{bundle.html_dir}`'))
if removed_html:
    display(Markdown(f'Removed previous notebook HTML files: `{len(removed_html)}`'))

dashboard_paths = {}
metric_paths = {}
selection_paths = {}
trend_paths = {}
representative_paths = []
forecast_paths = []
counterfactual_paths = []
mci_paths = []


## Existing Evaluation Metrics

In [ ]:
available_splits = direct_helpers.available_summary_splits(bundle)
display(Markdown('### Available evaluated splits'))
display(available_splits)

display(Markdown('### Overall split summary'))
display(direct_helpers.summary_table(bundle).round(6))

for split in available_splits:
    display(Markdown(f'### {split.upper()} diagnosis summary'))
    display(direct_helpers.diagnosis_summary_table(bundle, split).round(6))
    display(Markdown(f'### {split.upper()} pair-type summary'))
    display(direct_helpers.pair_type_summary_table(bundle, split).round(6))

gap_path = bundle.analysis_dir / 'gap_bin_summary.csv'
if gap_path.is_file():
    gap_summary = pd.read_csv(gap_path)
    gap_fig = rich.make_gap_bin_figure(gap_summary)
    metric_paths['gap_bin'] = direct_helpers.save_figure(bundle, gap_fig, 'gap_bin_performance')
    display(Markdown(f'### Gap-bin performance saved to `{metric_paths["gap_bin"]}`'))
    gap_fig.show()

dashboard_figures = direct_helpers.build_all_dashboard_figures(bundle)
for split, fig in dashboard_figures.items():
    path = direct_helpers.save_figure(bundle, fig, f'{split}_dashboard')
    dashboard_paths[split] = path
    display(Markdown(f'### {split.upper()} dashboard saved to `{path}`'))
    fig.show()


## Balanced Subject Selection

In [ ]:
selected_subjects = rich.select_balanced_subjects(
    bundle,
    subjects_per_split=SELECTED_SUBJECTS_PER_SPLIT,
    min_scans=MIN_SCANS_PER_SUBJECT,
)
selection_summary = rich.selection_summary_table(selected_subjects)

selection_tables = rich.save_tables(
    bundle.html_dir,
    {
        'selected_subjects_50_per_split': selected_subjects,
        'selected_subject_summary': selection_summary,
    },
)
selection_paths.update(selection_tables)

display(Markdown('### Selected-subject summary'))
display(selection_summary.round(4))
selection_fig = rich.make_selection_figure(selected_subjects)
selection_paths['selection_figure'] = direct_helpers.save_figure(bundle, selection_fig, 'selected_subjects_summary')
selection_fig.show()


## Volume, Longitudinal Prediction, And Speed Analysis

In [ ]:
if RUN_BATCH_VOLUME_DECODE:
    selected_volume_dataset = rich.build_selected_observed_age_volume_trend_dataset(
        bundle,
        selected_subjects,
        mesh_resolution=BATCH_MESH_RESOLUTION,
        mesh_max_batch=MESH_MAX_BATCH,
    )
    trend_frame = selected_volume_dataset['trend_frame']
    subject_volume_summary = selected_volume_dataset['subject_summary']
    decode_status = selected_volume_dataset['decode_status']
else:
    trend_frame = pd.read_csv(bundle.html_dir / 'selected_observed_age_volume_trend.csv')
    subject_volume_summary = pd.read_csv(bundle.html_dir / 'selected_subject_volume_summary.csv')
    decode_status = pd.read_csv(bundle.html_dir / 'selected_decode_status.csv')

speed_frame = rich.adjacent_volume_speed_frame(trend_frame)
speed_summary = rich.speed_summary_table(speed_frame)
prediction_consistency = rich.prediction_consistency_table(subject_volume_summary)

trend_tables = rich.save_tables(
    bundle.html_dir,
    {
        'selected_observed_age_volume_trend': trend_frame,
        'selected_subject_volume_summary': subject_volume_summary,
        'selected_decode_status': decode_status,
        'selected_adjacent_volume_speed': speed_frame,
        'selected_speed_summary': speed_summary,
        'selected_prediction_consistency': prediction_consistency,
    },
)
trend_paths.update(trend_tables)

display(Markdown('### Prediction consistency by split and diagnosis'))
display(prediction_consistency.round(6))
display(Markdown('### Adjacent volume-speed summary'))
display(speed_summary.round(6))

abs_age_fig = direct_helpers.observed_age_volume_trend_figure(trend_frame)
trend_paths['absolute_age_volume_html'] = direct_helpers.save_figure(bundle, abs_age_fig, 'selected_absolute_age_volume_trend')
abs_age_fig.show()

elapsed_fig = direct_helpers.elapsed_relative_volume_trend_figure(trend_frame)
trend_paths['elapsed_relative_volume_html'] = direct_helpers.save_figure(bundle, elapsed_fig, 'selected_elapsed_relative_volume_trend')
elapsed_fig.show()

delta_fig = rich.make_final_delta_comparison_figure(subject_volume_summary)
trend_paths['final_delta_html'] = direct_helpers.save_figure(bundle, delta_fig, 'selected_final_volume_delta_real_vs_prediction')
delta_fig.show()

speed_fig = rich.make_speed_box_figure(speed_frame)
trend_paths['speed_box_html'] = direct_helpers.save_figure(bundle, speed_fig, 'selected_volume_speed_real_vs_prediction')
speed_fig.show()

decode_failures = decode_status.loc[~decode_status['decode_success'].astype(bool)].copy()
display(Markdown('### Decode failures'))
if decode_failures.empty:
    display(Markdown('No decode failures in selected observed-age volume analysis.'))
else:
    display(decode_failures.head(50))


## Representative Interpolation Shape Pages

In [ ]:
representative_records = []
inline_count = 0

if RUN_REPRESENTATIVE_SHAPES:
    for split in ('train', 'val', 'test'):
        if split not in available_splits:
            continue
        for diagnosis in ('CN', 'AD'):
            try:
                pair_row = direct_helpers.select_representative_pair(
                    bundle,
                    split=split,
                    pair_type='nonadjacent',
                    diagnosis=diagnosis,
                    require_observed_intermediate=True,
                    max_source_age_years=min(OOD_AGES),
                )
                case = direct_helpers.build_pair_case(
                    bundle,
                    pair_row,
                    mesh_resolution=CASE_MESH_RESOLUTION,
                    mesh_max_batch=MESH_MAX_BATCH,
                )
                figures = {
                    'interpolation': direct_helpers.interpolation_figure(case),
                    'direct_vs_composed': direct_helpers.far_pair_direct_composed_figure(case),
                    'pair_volume': direct_helpers.pair_volume_trend_figure(case),
                }
                for name, fig in figures.items():
                    path = direct_helpers.save_figure(bundle, fig, f'{split}_{diagnosis.lower()}_{name}')
                    representative_paths.append(path)
                    if inline_count < INLINE_FIGURE_LIMIT:
                        fig.show()
                        inline_count += 1
                representative_records.append({
                    'split': split,
                    'diagnosis': diagnosis,
                    'subject_id': case['subject_id'],
                    'source_scan_id': case['source_row']['scan_id'],
                    'target_scan_id': case['target_row']['scan_id'],
                    'source_age_years': float(case['source_row']['continuous_age_years']),
                    'target_age_years': float(case['target_row']['continuous_age_years']),
                    'status': 'ok',
                })
            except Exception as exc:
                representative_records.append({
                    'split': split,
                    'diagnosis': diagnosis,
                    'subject_id': '',
                    'source_scan_id': '',
                    'target_scan_id': '',
                    'source_age_years': float('nan'),
                    'target_age_years': float('nan'),
                    'status': f'{type(exc).__name__}: {exc}',
                })

representative_summary = pd.DataFrame(representative_records)
trend_paths.update(rich.save_tables(bundle.html_dir, {'representative_interpolation_cases': representative_summary}))
display(Markdown('### Representative interpolation cases'))
display(representative_summary)


## Representative Observed And OOD Forecast Pages

In [ ]:
forecast_records = []
inline_count = 0

if RUN_REPRESENTATIVE_SHAPES:
    for split in ('train', 'val', 'test'):
        if split not in available_splits:
            continue
        for diagnosis in ('CN', 'AD'):
            try:
                pair_row = direct_helpers.select_representative_pair(
                    bundle,
                    split=split,
                    pair_type='nonadjacent',
                    diagnosis=diagnosis,
                    require_observed_intermediate=True,
                    max_source_age_years=min(OOD_AGES),
                )
                case = direct_helpers.build_subject_forecast_case(
                    bundle,
                    pair_row,
                    ood_ages_years=OOD_AGES,
                    composed_step_years=COMPOSED_STEP_YEARS,
                    mesh_resolution=CASE_MESH_RESOLUTION,
                    mesh_max_batch=MESH_MAX_BATCH,
                )
                volume_fig = direct_helpers.subject_forecast_volume_figure(case)
                volume_path = direct_helpers.save_figure(bundle, volume_fig, f'{split}_{diagnosis.lower()}_subject_volume_forecast')
                forecast_paths.append(volume_path)
                if inline_count < INLINE_FIGURE_LIMIT:
                    volume_fig.show()
                    inline_count += 1

                observed_fig, observed_summary = direct_helpers.observed_target_change_heatmap_figure(
                    case,
                    sample_count=CHANGE_SAMPLE_COUNT,
                )
                observed_path = direct_helpers.save_figure(bundle, observed_fig, f'{split}_{diagnosis.lower()}_observed_target_change')
                forecast_paths.append(observed_path)

                if case['ood_ages_years']:
                    direct_ood_fig, direct_ood_summary = direct_helpers.ood_change_heatmap_figure(
                        case,
                        method='direct',
                        sample_count=CHANGE_SAMPLE_COUNT,
                    )
                    direct_ood_path = direct_helpers.save_figure(bundle, direct_ood_fig, f'{split}_{diagnosis.lower()}_ood_direct_change')
                    forecast_paths.append(direct_ood_path)

                    composed_ood_fig, composed_ood_summary = direct_helpers.ood_change_heatmap_figure(
                        case,
                        method='composed',
                        sample_count=CHANGE_SAMPLE_COUNT,
                    )
                    composed_ood_path = direct_helpers.save_figure(bundle, composed_ood_fig, f'{split}_{diagnosis.lower()}_ood_composed_change')
                    forecast_paths.append(composed_ood_path)
                else:
                    direct_ood_summary = pd.DataFrame()
                    composed_ood_summary = pd.DataFrame()

                forecast_records.append({
                    'split': split,
                    'diagnosis': diagnosis,
                    'subject_id': case['subject_id'],
                    'source_scan_id': case['source_row']['scan_id'],
                    'source_age_years': float(case['source_age_years']),
                    'last_real_age_years': float(case['last_real_age_years']),
                    'ood_ages_years': ','.join(f'{age:.1f}' for age in case['ood_ages_years']),
                    'observed_mean_surface_shift': float(observed_summary['mean_surface_shift'].mean()) if not observed_summary.empty else float('nan'),
                    'direct_ood_mean_surface_shift': float(direct_ood_summary['mean_surface_shift'].mean()) if not direct_ood_summary.empty else float('nan'),
                    'composed_ood_mean_surface_shift': float(composed_ood_summary['mean_surface_shift'].mean()) if not composed_ood_summary.empty else float('nan'),
                    'status': 'ok',
                })
            except Exception as exc:
                forecast_records.append({
                    'split': split,
                    'diagnosis': diagnosis,
                    'subject_id': '',
                    'source_scan_id': '',
                    'source_age_years': float('nan'),
                    'last_real_age_years': float('nan'),
                    'ood_ages_years': '',
                    'observed_mean_surface_shift': float('nan'),
                    'direct_ood_mean_surface_shift': float('nan'),
                    'composed_ood_mean_surface_shift': float('nan'),
                    'status': f'{type(exc).__name__}: {exc}',
                })

forecast_summary = pd.DataFrame(forecast_records)
trend_paths.update(rich.save_tables(bundle.html_dir, {'representative_forecast_cases': forecast_summary}))
display(Markdown('### Representative forecast and OOD cases'))
display(forecast_summary)


## Counterfactual Condition Forecasts

In [ ]:
counterfactual_records = []
counterfactual_sources = rich.select_counterfactual_source_rows(bundle, selected_subjects, per_split_per_diagnosis=1)
inline_count = 0

if RUN_REPRESENTATIVE_SHAPES:
    for _, source in counterfactual_sources.iterrows():
        try:
            case = rich.build_counterfactual_condition_case(
                bundle,
                split=str(source['split']),
                subject_id=str(source['subject_id']),
                horizon_years=COUNTERFACTUAL_HORIZON_YEARS,
                evaluation_step_years=COUNTERFACTUAL_STEP_YEARS,
                mesh_resolution=COUNTERFACTUAL_MESH_RESOLUTION,
                mesh_max_batch=MESH_MAX_BATCH,
            )
            volume_fig = rich.counterfactual_volume_figure(case)
            volume_path = direct_helpers.save_figure(
                bundle,
                volume_fig,
                f'{case["split"]}_{case["source_diagnosis"].lower()}_{case["subject_id"]}_counterfactual_volume',
            )
            counterfactual_paths.append(volume_path)
            if inline_count < INLINE_FIGURE_LIMIT:
                volume_fig.show()
                inline_count += 1

            change_fig, change_summary = rich.counterfactual_final_change_figure(
                case,
                sample_count=CHANGE_SAMPLE_COUNT,
            )
            change_path = direct_helpers.save_figure(
                bundle,
                change_fig,
                f'{case["split"]}_{case["source_diagnosis"].lower()}_{case["subject_id"]}_counterfactual_final_change',
            )
            counterfactual_paths.append(change_path)

            status = case['decode_status']
            volume_by_condition = case['trend_frame'].sort_values('age_years').groupby('condition_name').tail(1)
            cn_final = volume_by_condition.loc[volume_by_condition['condition_name'] == 'CN_condition', 'volume']
            ad_final = volume_by_condition.loc[volume_by_condition['condition_name'] == 'AD_condition', 'volume']
            counterfactual_records.append({
                'split': case['split'],
                'subject_id': case['subject_id'],
                'source_diagnosis': case['source_diagnosis'],
                'source_age_years': float(case['source_age_years']),
                'final_age_years': float(case['final_age_years']),
                'cn_condition_final_volume': float(cn_final.iloc[0]) if not cn_final.empty else float('nan'),
                'ad_condition_final_volume': float(ad_final.iloc[0]) if not ad_final.empty else float('nan'),
                'decode_success_fraction': float(status['decode_success'].astype(bool).mean()) if not status.empty else float('nan'),
                'status': 'ok',
            })
        except Exception as exc:
            counterfactual_records.append({
                'split': str(source.get('split', '')),
                'subject_id': str(source.get('subject_id', '')),
                'source_diagnosis': str(source.get('diagnosis', '')),
                'source_age_years': float('nan'),
                'final_age_years': float('nan'),
                'cn_condition_final_volume': float('nan'),
                'ad_condition_final_volume': float('nan'),
                'decode_success_fraction': float('nan'),
                'status': f'{type(exc).__name__}: {exc}',
            })

counterfactual_summary = pd.DataFrame(counterfactual_records)
trend_paths.update(rich.save_tables(bundle.html_dir, {'counterfactual_condition_summary': counterfactual_summary}))
display(Markdown('### Counterfactual condition summary'))
display(counterfactual_summary.round(6))


## With-MCI Ground-Truth Reference Trend

In [ ]:
if RUN_MCI_REFERENCE:
    with_mci_root = REPO / 'examples' / 'ADNI_1_L_With_MCI' / 'brainode_comparison_task1_manifest_original'
    with_mci = mci_helpers.load_manifest(
        with_mci_root,
        mci_helpers.WITH_MCI_PREFIX,
        name='ADNI original with-MCI',
    )
    mci_visit_df = mci_helpers.build_visit_dataframe(with_mci.clean_df)
    mci_pair_df = mci_helpers.build_adjacent_pair_dataframe(with_mci.clean_df)
    mci_subject_df = mci_helpers.subject_mean_speed_dataframe(mci_pair_df)
    mci_hotspot = mci_helpers.shape_hotspot_summary_table(with_mci.clean_df, ['CN', 'MCI', 'AD'], top_fraction=0.10, focus='inward')
    mci_similarity = mci_helpers.pairwise_shape_similarity_table(with_mci.clean_df, ['CN', 'MCI', 'AD'], top_fraction=0.10, focus='inward')

    mci_tables = rich.save_tables(
        bundle.html_dir,
        {
            'mci_reference_visit_volume': mci_visit_df,
            'mci_reference_adjacent_speed': mci_pair_df,
            'mci_reference_subject_speed': mci_subject_df,
            'mci_reference_hotspot_summary': mci_hotspot,
            'mci_reference_hotspot_similarity': mci_similarity,
        },
    )
    trend_paths.update(mci_tables)
    display(Markdown('### With-MCI reference hotspot summary'))
    display(mci_hotspot.round(6))
    display(Markdown('### With-MCI reference hotspot similarity'))
    display(mci_similarity.round(6))

    comparison_fig = rich.make_mci_reference_speed_comparison_figure(speed_frame, mci_pair_df)
    comparison_path = direct_helpers.save_figure(bundle, comparison_fig, 'mci_reference_speed_comparison')
    mci_paths.append(comparison_path)
    comparison_fig.show()

    mci_figures = []
    mci_figures.extend(mci_helpers.step2_figures(with_mci))
    mci_figures.extend(mci_helpers.common_longitudinal_figures(with_mci, ['CN', 'MCI', 'AD'], 'with-MCI'))
    mci_figures.extend(mci_helpers.local_shape_figures(with_mci.clean_df, 'ADNI original with-MCI', ['CN', 'MCI', 'AD']))
    mci_figures.extend(mci_helpers.shape_difference_figures(with_mci.clean_df, 'ADNI original with-MCI', ['CN', 'MCI', 'AD'], [('CN', 'MCI'), ('CN', 'AD'), ('MCI', 'AD')]))

    for idx, fig in enumerate(mci_figures, start=1):
        path = direct_helpers.save_figure(bundle, fig, f'mci_reference_{idx:02d}')
        mci_paths.append(path)
else:
    display(Markdown('With-MCI reference section skipped.'))


## Output Index

In [ ]:
index_path = rich.write_html_index(
    bundle.html_dir,
    title='Large Strict Direct-Flow Rich Analysis',
    sections=[
        ('Metric dashboards', list(dashboard_paths.values()) + list(metric_paths.values())),
        ('Subject selection and volume trend', [p for p in selection_paths.values() if str(p).endswith('.html')] + [p for p in trend_paths.values() if str(p).endswith('.html')]),
        ('Representative interpolation pages', representative_paths),
        ('Representative observed and OOD forecast pages', forecast_paths),
        ('Counterfactual CN-vs-AD condition pages', counterfactual_paths),
        ('With-MCI ground-truth reference pages', mci_paths),
    ],
)
display(Markdown(f'### HTML index saved to `{index_path}`'))
display(Markdown('### CSV tables saved in the same notebook output directory'))
for path in sorted(bundle.html_dir.glob('*.csv')):
    print(path)
display(Markdown('### HTML files'))
for path in sorted(bundle.html_dir.glob('*.html')):
    print(path)
